# NB_08_B_MEASURED_ENGINEERING_STATE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkthoughts/sensors-becker/blob/main/notebooks/NB_08_B_MEASURED_ENGINEERING_STATE.ipynb)

This notebook continues RP_08 by inheriting Candidate A and realizing the Measurement → Measured Engineering State relationship.


In [ ]:
NOTEBOOK_ID = "NB_08_B_MEASURED_ENGINEERING_STATE"
NOTEBOOK_FILENAME = f"{NOTEBOOK_ID}.ipynb"
NOTEBOOK_VERSION = "0.2.0"
ENGINEERING_STAGE = "B"
RELEASE_FILENAME = f"{NOTEBOOK_ID}.zip"

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "architecture": "specification-driven",
    "status": "candidate",
    "release_filename": RELEASE_FILENAME,
}


## Initialize Repository Runtime

The notebook initializes the repository once. All candidate artifacts derive from the Reading Point and notebook specifications.


In [ ]:
from __future__ import annotations

import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
COLAB_REPOSITORY_ROOT = Path("/content/sensors-becker")


def install_colab_repository() -> Path:
    if COLAB_REPOSITORY_ROOT.exists():
        shutil.rmtree(COLAB_REPOSITORY_ROOT)

    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(COLAB_REPOSITORY_ROOT)],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(COLAB_REPOSITORY_ROOT),
        ],
        check=True,
    )

    src_directory = COLAB_REPOSITORY_ROOT / "src"
    if str(src_directory) not in sys.path:
        sys.path.insert(0, str(src_directory))

    importlib.invalidate_caches()
    os.chdir(COLAB_REPOSITORY_ROOT)
    return COLAB_REPOSITORY_ROOT


try:
    import sensors_becker
except ModuleNotFoundError:
    repository_root = install_colab_repository()
    import sensors_becker
else:
    repository_root = Path(sensors_becker.__file__).resolve().parents[2]

from sensors_becker import initialize_notebook

runtime = initialize_notebook(
    start=repository_root,
    environment=(
        "google-colab"
        if repository_root == COLAB_REPOSITORY_ROOT
        else "repository-runtime"
    ),
)
context = runtime.context
runtime.validate()

print(f"Environment: {runtime.environment}")
print(f"Repository root: {runtime.repository_root}")
print("Engineering context validation: PASSED")


## Initialize Reading Point RP_08

RP_08 inherits Candidate A and develops a second measured-engineering-states relationship.


In [ ]:
READING_POINT = {
    "reading_point_id": "RP_08",
    "inherited_from": "NB_08_A_MEASURED_ENGINEERING_STATES",
    "inherited_engineering_dialogue": (
        "Engineering System",
        "Measured Engineering States",
    ),
    "natural_foundation": "Measured Engineering States",
    "engineering_objective": "Specify Measured Engineering States.",
    "engineering_statements": (
        "Engineering System produces Measured Engineering States.",
    ),
    "candidate_engineering_statements": (
        "Measurement records Measured Engineering State.",
    ),
    "observed_contrast": (),
    "engineering_candidates": (
        "Measured Engineering State",
    ),
    "specification_search": {
        "candidate_pair": (
            "Measurement",
            "Measured Engineering State",
        ),
        "forward_statement": (
            "Measured Engineering State informs Leading Specification."
        ),
        "forward_context": "Leading Specification",
        "status": "developing",
    },
    "repository_grammar": (
        "Engineering Object specifies Engineering System.",
        "Engineering System produces Measured Engineering States.",
        "Measurement records Measured Engineering State.",
        "Measured Engineering State informs Leading Specification.",
        "Measured Engineering States identify Engineering Constraints.",
        "Engineering Constraints direct Engineering Refinements.",
        "Engineering Refinements support Measured Engineering Improvement.",
        "Measured Engineering Improvement informs Leading Specifications.",
    ),
    "admitted_engineering_dialogue": (
        "Engineering System",
        "Measured Engineering States",
    ),
}

assert READING_POINT["natural_foundation"] == "Measured Engineering States"
assert READING_POINT["engineering_candidates"] == ("Measured Engineering State",)
assert READING_POINT["candidate_engineering_statements"] == (
    "Measurement records Measured Engineering State.",
)

READING_POINT


## Admit Candidate Notebook Specification

Candidate B inherits Candidate A and realizes one new engineering relationship:

**Measurement records Measured Engineering State.**


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class DialogueArtifactSpec:
    order: str
    artifact_id: str
    concept: str
    title: str
    first_label: str
    second_label: str
    supporting_context: tuple[str, str]
    engineering_statement: str
    status: str


NOTEBOOK_SPEC = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_scope": "measured_engineering_state",
    "engineering_object": "Microcalorimeter",
    "engineering_direction": "Toward next-generation microcalorimeters.",
    "footer": "Admissible generalizations trail leading specifications.",
    "renderer_version": "3",
    "dialogue": (
        DialogueArtifactSpec(
            order="A",
            artifact_id="08_A_measured_engineering_states_trail",
            concept="Measured Engineering States",
            title="Measured Engineering States Trail: Microcalorimeters",
            first_label="Engineering System",
            second_label="Measured Engineering States",
            supporting_context=("Measurement", "Leading Specification"),
            engineering_statement="Engineering System produces Measured Engineering States.",
            status="admitted",
        ),
        DialogueArtifactSpec(
            order="B",
            artifact_id="08_B_measured_engineering_state_trail",
            concept="Measured Engineering State",
            title="Measured Engineering State Trail: Microcalorimeters",
            first_label="Measurement",
            second_label="Measured Engineering State",
            supporting_context=("Engineering System", "Leading Specification"),
            engineering_statement="Measurement records Measured Engineering State.",
            status="specification-search",
        ),
    ),
}

assert tuple(item.order for item in NOTEBOOK_SPEC["dialogue"]) == ("A", "B")
assert NOTEBOOK_SPEC["dialogue"][0].status == "admitted"
assert NOTEBOOK_SPEC["dialogue"][1].status == "specification-search"
assert NOTEBOOK_SPEC["dialogue"][1].engineering_statement == READING_POINT["candidate_engineering_statements"][0]

NOTEBOOK_SPEC


## Prepare Specification Renderer

One generic renderer realizes the admitted Candidate A figure and the Candidate B measured-engineering-state figure.


In [ ]:
import json
import zipfile
from datetime import date
from hashlib import sha256

from IPython.display import Image, Markdown, display

from sensors_becker.dialogue_renderer import (
    DialogueFigure,
    DialogueNode,
    DialogueRelation,
    NotebookDialogueRenderer,
)

renderer = NotebookDialogueRenderer(
    figsize=(12, 8),
    dpi=180,
    validate_layout=True,
)


def sha256_for_path(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def display_artifact(path: Path) -> None:
    if path.suffix.lower() == ".png":
        display(Image(filename=str(path)))
    elif path.suffix.lower() == ".md":
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print(path.read_text(encoding="utf-8"))
    print(f"Artifact: {path.name}")


def build_figure(spec: DialogueArtifactSpec) -> DialogueFigure:
    return DialogueFigure(
        title=spec.title,
        subtitle=NOTEBOOK_SPEC["engineering_direction"],
        footer=NOTEBOOK_SPEC["footer"],
        primary_nodes=(
            DialogueNode(
                label=spec.first_label,
                x=0.5,
                y=0.67,
                width=0.42,
                height=0.10,
                role="input",
                fontsize=18,
            ),
            DialogueNode(
                label=spec.second_label,
                x=0.5,
                y=0.46,
                width=0.42,
                height=0.12,
                role="primary",
                emphasis=True,
                fontsize=21,
            ),
        ),
        primary_relations=(
            DialogueRelation(
                start=(0.5, 0.62),
                end=(0.5, 0.52),
                role="input",
                directional=True,
                linewidth=2.0,
            ),
        ),
        supporting_context=spec.supporting_context,
    )


def build_alt_text(spec: DialogueArtifactSpec) -> str:
    left, right = spec.supporting_context
    return (
        "# Alt Text\n\n"
        f"**{spec.title}.** Diagram showing {spec.first_label.lower()} "
        f"leading to {spec.second_label.lower()}. "
        f"{left} and {right} appear as supporting engineering dialogue. "
        f"The candidate engineering statement is: “{spec.engineering_statement}” "
        f"Subtitle: “{NOTEBOOK_SPEC['engineering_direction']}” "
        f"Footer: “{NOTEBOOK_SPEC['footer']}”\n"
    )


bundle_directory = runtime.paths.outputs / "releases" / NOTEBOOK_ID
bundle_directory.mkdir(parents=True, exist_ok=True)

for prior_path in bundle_directory.iterdir():
    if prior_path.is_file():
        prior_path.unlink()

print(f"Renderer: {renderer.__class__.__name__}")
print("Architecture: specification-driven")
print("Reading Point: RP_07")
print(f"Candidate artifact specifications: {len(NOTEBOOK_SPEC['dialogue'])}")
print(f"Bundle directory: {runtime.relative_path(bundle_directory)}")


## Generate Candidate Bundle

This operation realizes:

- **08_A** — admitted inherited dialogue.
- **08_B** — current measured-engineering-state specification search.


In [ ]:
generated_records = []

for artifact_spec in NOTEBOOK_SPEC["dialogue"]:
    png_path = bundle_directory / f"{artifact_spec.artifact_id}.png"
    alt_path = bundle_directory / f"{artifact_spec.artifact_id}.alt.md"

    renderer.render(build_figure(artifact_spec), png_path)
    alt_path.write_text(build_alt_text(artifact_spec), encoding="utf-8")

    generated_records.append({"spec": artifact_spec, "png_path": png_path, "alt_path": alt_path})
    display_artifact(png_path)
    display_artifact(alt_path)

assert len(generated_records) == 2
assert generated_records[0]["spec"].status == "admitted"
assert generated_records[1]["spec"].status == "specification-search"


## Record Candidate Bundle

README, metadata, and manifest records derive from RP_08 and both Candidate A and Candidate B artifacts.


In [ ]:
readme_path = bundle_directory / "08_C_README.md"
metadata_path = bundle_directory / "08_D_notebook_metadata.json"
manifest_path = bundle_directory / "08_E_manifest.json"

artifact_a, artifact_b = NOTEBOOK_SPEC["dialogue"]
repository_grammar_lines = "\n".join(f"- {statement}" for statement in READING_POINT["repository_grammar"])

readme_text = f"""# {NOTEBOOK_ID}

This candidate bundle continues RP_08 from Candidate A.

## Natural Foundation

{READING_POINT["natural_foundation"]}

## Engineering Objective

{READING_POINT["engineering_objective"]}

## Admitted Engineering Statement

{artifact_a.engineering_statement}

## Candidate B Engineering Statement

{artifact_b.engineering_statement}

## Candidate B Dialogue

{artifact_b.first_label}

↓

{artifact_b.second_label}

## Supporting Context

{artifact_b.supporting_context[0]}

{artifact_b.supporting_context[1]}

## Forward Specification Search

{READING_POINT["specification_search"]["forward_statement"]}

## Repository Grammar

{repository_grammar_lines}

## Engineering Object

{NOTEBOOK_SPEC["engineering_object"]}

## Engineering Direction

{NOTEBOOK_SPEC["engineering_direction"]}

---

*{NOTEBOOK_SPEC["footer"]}*
"""
readme_path.write_text(readme_text, encoding="utf-8")

reading_point_record = {
    "reading_point_id": READING_POINT["reading_point_id"],
    "inherited_from": READING_POINT["inherited_from"],
    "inherited_engineering_dialogue": list(READING_POINT["inherited_engineering_dialogue"]),
    "natural_foundation": READING_POINT["natural_foundation"],
    "engineering_objective": READING_POINT["engineering_objective"],
    "engineering_statements": list(READING_POINT["engineering_statements"]),
    "candidate_engineering_statements": list(READING_POINT["candidate_engineering_statements"]),
    "observed_contrast": list(READING_POINT["observed_contrast"]),
    "engineering_candidates": list(READING_POINT["engineering_candidates"]),
    "specification_search": {
        "candidate_pair": list(READING_POINT["specification_search"]["candidate_pair"]),
        "forward_statement": READING_POINT["specification_search"]["forward_statement"],
        "forward_context": READING_POINT["specification_search"]["forward_context"],
        "status": READING_POINT["specification_search"]["status"],
    },
    "repository_grammar": list(READING_POINT["repository_grammar"]),
    "admitted_engineering_dialogue": list(READING_POINT["admitted_engineering_dialogue"]),
}

metadata = {
    "artifact_type": "candidate_notebook_bundle",
    "notebook_id": NOTEBOOK_ID,
    "notebook_filename": NOTEBOOK_FILENAME,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": context.repository,
    "engineering_stage": ENGINEERING_STAGE,
    "engineering_scope": NOTEBOOK_SPEC["engineering_scope"],
    "engineering_object": NOTEBOOK_SPEC["engineering_object"],
    "engineering_direction": NOTEBOOK_SPEC["engineering_direction"],
    "architecture": "specification-driven",
    "status": "candidate",
    "reading_point": reading_point_record,
    "renderer": {"name": renderer.__class__.__name__, "version": NOTEBOOK_SPEC["renderer_version"]},
    "runtime_environment": runtime.environment,
    "generated": date.today().isoformat(),
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

manifest_artifacts = []
for record in generated_records:
    item = record["spec"]
    for realization_order, path in enumerate((record["png_path"], record["alt_path"]), start=1):
        manifest_artifacts.append({
            "artifact_order": item.order,
            "artifact_id": item.artifact_id,
            "engineering_concept": item.concept,
            "engineering_statement": item.engineering_statement,
            "dialogue_status": item.status,
            "primary_dialogue": [item.first_label, item.second_label],
            "supporting_context": list(item.supporting_context),
            "realization_order": realization_order,
            "realization_type": "png" if path.suffix.lower() == ".png" else "alt_text",
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_for_path(path),
            "status": "verified",
        })
for order, path in (("C", readme_path), ("D", metadata_path)):
    manifest_artifacts.append({
        "artifact_order": order,
        "artifact_id": path.stem,
        "engineering_concept": "bundle_record",
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_for_path(path),
        "status": "verified",
    })
manifest = {
    "release_filename": RELEASE_FILENAME,
    "repository": context.repository,
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "architecture": "specification-driven",
    "status": "candidate",
    "reading_point": reading_point_record,
    "dialogue": [{"order": i.order, "concept": i.concept, "statement": i.engineering_statement, "status": i.status} for i in NOTEBOOK_SPEC["dialogue"]],
    "artifacts": manifest_artifacts,
}
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
for path in (readme_path, metadata_path, manifest_path):
    display_artifact(path)


## Verify Candidate Notebook

Verification compares both dialogue artifacts with RP_08, the admitted Candidate A statement, and the Candidate B measured-engineering-state statement.


In [ ]:
bundle_artifact_paths = [
    path for record in generated_records for path in (record["png_path"], record["alt_path"])
] + [readme_path, metadata_path, manifest_path]

expected_names = [
    "08_A_measured_engineering_states_trail.png",
    "08_A_measured_engineering_states_trail.alt.md",
    "08_B_measured_engineering_state_trail.png",
    "08_B_measured_engineering_state_trail.alt.md",
    "08_C_README.md",
    "08_D_notebook_metadata.json",
    "08_E_manifest.json",
]
assert [path.name for path in bundle_artifact_paths] == expected_names
assert all(path.exists() and path.stat().st_size > 0 for path in bundle_artifact_paths)

with metadata_path.open(encoding="utf-8") as handle:
    metadata_check = json.load(handle)
with manifest_path.open(encoding="utf-8") as handle:
    manifest_check = json.load(handle)

assert metadata_check["reading_point"]["reading_point_id"] == "RP_08"
assert metadata_check["reading_point"]["inherited_from"] == "NB_08_A_MEASURED_ENGINEERING_STATES"
assert metadata_check["reading_point"]["engineering_objective"] == "Specify Measured Engineering States."
assert metadata_check["reading_point"]["candidate_engineering_statements"] == ["Measurement records Measured Engineering State."]
assert metadata_check["reading_point"]["specification_search"]["forward_statement"] == "Measured Engineering State informs Leading Specification."
assert manifest_check["dialogue"][0]["status"] == "admitted"
assert manifest_check["dialogue"][1]["status"] == "specification-search"
assert manifest_check["dialogue"][1]["statement"] == "Measurement records Measured Engineering State."
for record in manifest_check["artifacts"]:
    path = bundle_directory / record["filename"]
    assert path.exists()
    assert sha256_for_path(path) == record["sha256"]
for path in bundle_artifact_paths:
    print(f"✓ {path.name} ({path.stat().st_size} bytes)")
print("✓ RP_08 inheritance verified")
print("✓ Candidate A admitted dialogue verified")
print("✓ Candidate B measured-engineering-state search verified")
print("✓ Forward specification search recorded")
print("✓ Complete candidate bundle verified")


## Package and Download

The complete verified RP_08 Candidate B bundle is packaged in reading order.


In [ ]:
release_directory = runtime.paths.outputs / "releases"
release_directory.mkdir(parents=True, exist_ok=True)
release_path = release_directory / RELEASE_FILENAME

with zipfile.ZipFile(release_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_artifact_paths:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(release_path) as archive:
    assert archive.namelist() == expected_names

print(
    f"✓ {runtime.relative_path(release_path)} "
    f"({release_path.stat().st_size} bytes)"
)

if runtime.environment == "google-colab":
    from google.colab import files
    files.download(str(release_path))
else:
    print(release_path)


*Admissible generalizations trail leading specifications.*